# Laboratorio 3

## Cálculo de Accesibilidades Gravitacionales

Ayudante: Janus Leonhardt | jaleonhardt@uc.cl  

Profesor: Ricardo Hurtubia | rhurtubia@uc.cl

## Objetivos del Laboratorio:

1. Consolidar la información de las celdas enriquecida con datos catastrales, censales y paraderos para calcular accesibilidades gravitacionales.

2. Explicar el concepto de la función de impedancia y cómo se aplica en el cálculo de accesibilidades gravitacionales.

3. Introducir herramientas para generar matrices de costo/tiempo/distancia entre celdas (p.ej., OSRM), y ejemplificar cómo integrar dichos valores en el modelo de accesibilidad gravitacional.

4. Guardar los resultados de la estimación de accesibilidades en un archivo GeoJSON para su uso posterior.

---

### 1. Repaso del Laboratorio 2

En el **Laboratorio 2**, aprendimos sobre la integración de información geoespacial a nuestras celdas, a partir de los métodos *overlay* y *sjoin* de GeoPandas, creando un GeoDataFrame de celdas con variables como `personas`, `hogares`, `n_paraderos`, etc. A partir de esta base de datos consolidada, en el Laboratorio 3 nos centraremos en el **cálculo de accesibilidades gravitacionales**.

---

## 2. Instalación y Configuración del Entorno de Trabajo

### Requisitos Previos

- Python 3.7 o superior

- Jupyter Notebook

- **Librerías de Python**:

  - pandas  
  - geopandas  
  - shapely  
  - h3  
  - matplotlib  
  - folium  
  - mapclassify  
  - seaborn
  - **requests**

### Instalación de Librerías

Recordemos que las librerías deben haber quedado correctamente instaladas en nuestro entorno virtual (`.conda` o `.venv`) creado en el Laboratorio 1. En caso contrario, se debe ejecutar la siguiente línea de código.

`!pip install pandas geopandas shapely h3 matplotlib folium mapclassify seaborn requests`

In [ ]:
# instalar las librerías faltantes
!pip install requests

---

## 3. Carga de Datos Previos

En este laboratorio, partiremos desde los archivos generados en el Laboratorio 2, los cuales contienen la información catastral relacionada con las celdas y sus atributos del entorno construido que integramos de fuentes de datos externas (Censo 2017, SIEDU).

- **`celdas.geojson`**: Contiene información geoespacial agregada por cada celda, incluyendo la geometría hexagonal y atributos del entorno construido que se obtienen de la agregación de las líneas de construcción y de otras fuentes de datos.

In [ ]:
import pandas as pd
import geopandas as gpd

# ajustes de visualización
pd.set_option('display.max_columns', None)

# cargar la capa de celdas con información agregada
celdas = gpd.read_file("../lab_02/celdas.geojson")

In [ ]:
# revisamos el sistema de referencia de coordenadas
print(celdas.crs)

Recordar que lo ideal es que nuestros GeoDataFrame se encuentren en WGS84 (EPSG:4326) si vamos a mostrar mapas interactivos, o en un CRS proyectado (EPSG:32719) si calcularemos distancias en metros.

---

## 4. Introducción al Modelo Gravitacional de Accesibilidad

El modelo gravitacional de accesibilidad asume que la atracción de oportunidades (p. ej., población, empleos, comercio) decrece con el costo/distancia/tiempo para llegar a ellas. Sean $i, j$ las celdas de origen y destino, $m$ el modo de transporte (p. ej, automóvil, caminata, transporte público), $p$ el propósito/tipo de actividad/oportunidad (en este caso relacionadas con el uso de suelo), $N_{j}^{p}$ la cantidad/calidad de oportunidades de tipo $p$ en el celda $j$ y $F_{i,j}^{m}$ el valor de la función de impedancia que aplica entre la celda $i$ a $j$ en el modo $m$, entonces la accesibilidad de la celda $i$ en el modo $m$ a las oportunidades del tipo $p$ queda definida de la siguiente manera.


$acc_{i}^{m,p} = \sum_{j=1}^{J} N_{j}^{p} \cdot F_{i,j}^{m}$

### Ejemplo de Función de Impedancia (Exponencial)

La función de impedancia es un componente clave en el cálculo del modelo gravitacional de accesibilidad, ya que mide cómo disminuye la atracción de oportunidades en función del costo generalizado (e.g., tiempo, distancia o gasto) para alcanzarlas. Una de las formas más comunes de representar esta relación es mediante la función exponencial, definida a continuación.


$F_{i,j}^{m} = e^{\phi \cdot C_{i,j}^{m}}$


En este caso, $C_{i,j}^{m}$ representa el costo generalizado para viajar entre la celda $i$ y la celda $j$ en el modo de transporte $m$ (p. ej, distancia, tiempo o costo monetario del viaje) y $\phi$ es un parámetro que ajusta el decaimiento (a mayor $\phi$ , el costo “penaliza” más la accesibilidad).

---

## 5. Generación de una Matriz de Costos

Para estimar $acc_{i}^{m,p}$, necesitamos $C_{i,j}^{m}$, , es decir, la **distancia o tiempo** de viaje entre las celdas $i$ y $j$. En este laboratorio, construiremos la **matriz de tiempos de viaje** (o matriz Origen-Destino) entre todas las celdas para el modo automóvil y caminata, utilizando [**Open Source Routing Machine (OSRM)**](https://github.com/Project-OSRM/osrm-backend).

### Breve Introducción a OSRM

**Open Source Routing Machine (OSRM)** es una herramienta de código abierto que permite preprocesar la red vial (o de caminata) de **OpenStreetMap** y ofrece un *servicio web* para consultas de enrutamiento (rutas, tiempos) y matrices de tiempos de viaje.

Lo configuramos localmente mediante [**Docker**](https://www.docker.com), corriendo dos contenedores OSRM (uno para el modo automóvil, otro para el modo caminata). Esto es especialmente útil para zonas de estudio grandes, donde herramientas como `osmnx` y `networkx` pueden volverse ineficientes al crear matrices OD de muchas celdas.

### Configurar OSRM Local con Docker

**1. Instalar Docker**

Para usar OSRM localmente, primero necesitas instalar Docker en tu máquina. Sigue las instrucciones a continuación según tu sistema operativo.

- **Para Windows**:  
  Descarga e instala Docker Desktop desde [este enlace](https://docs.docker.com/desktop/install/windows-install/). Después de la instalación, asegúrate de que Docker esté funcionando abriendo Docker Desktop.

- **Para macOS**:  
  Descarga e instala Docker Desktop desde [este enlace](https://docs.docker.com/desktop/install/mac-install/). Una vez instalado, puedes abrir Docker Desktop desde tu carpeta de Aplicaciones.

- **Para Linux**:  
  Instala Docker siguiendo las instrucciones oficiales para tu distribución desde [este enlace](https://docs.docker.com/desktop/install/linux/). Asegúrate de habilitar y ejecutar Docker después de la instalación.

Una vez que Docker esté instalado y corriendo, ya puedes proceder con la configuración de OSRM.

**2. Descargar los Archivos de OSRM**

Debes [descargar los archivos](https://drive.google.com/drive/folders/1C8E5Dt-ol-C1578up6g4Fh9Z5cjRomiG?usp=share_link) necesarios para correr OSRM en tu computador. Descarga y descomprime los archivos en una carpeta local en tu máquina.

**3. Correr OSRM para Modo Automóvil y Caminata**

Para correr OSRM, usa los siguientes comandos en tu Shell/Terminal. Debes ajustar la ruta de la carpeta donde guardaste los archivos descargados.

  **OSRM para Modo Automóvil**

  Abre el Terminal y ejecuta el siguiente comando (cambia la ruta `/ruta/a/tu/carpeta/OSRMData_Drive` por la ruta donde descargaste los datos):

  ```bash
  docker run -t -i -p 5001:5000 -v "/ruta/a/tu/carpeta/OSRMData_Drive:/data" osrm/osrm-backend osrm-routed --algorithm ch --max-table-size 1000000 /data/chile-latest.osrm --mmap=0
  ```

  Este comando correrá el servidor de OSRM para rutas en automóvil en el puerto 5001.

  **OSRM para Modo Caminata**

  Ahora, para rutas en caminata, ejecuta el siguiente comando en el Terminal (cambia la ruta `/ruta/a/tu/carpeta/OSRMData_Walk` por la ubicación correcta):

  ```bash
  docker run -t -i -p 5002:5000 -v "/ruta/a/tu/carpeta/OSRMData_Walk:/data" osrm/osrm-backend osrm-routed --algorithm ch --max-table-size 1000000 /data/chile-latest.osrm --mmap=0
  ```
  Esto correrá el servidor de OSRM para rutas en caminata en el puerto 5002.

**¿Cómo Probar?**

Después de ejecutar estos comandos, los servidores de OSRM estarán activos en tu máquina.

Modo Automóvil: El servidor estará disponible en http://localhost:5001

Modo Caminata: El servidor estará disponible en http://localhost:5002

Puedes probarlos enviando solicitudes desde tu navegador o con alguna herramienta como [Postman](https://www.postman.com).

**Notas Importantes**: *Asegúrate de tener Docker corriendo cada vez que quieras utilizar OSRM. Si tienes problemas con los puertos, asegúrate de que no estén siendo utilizados por otros servicios.*

### Obtención de la Matriz Origen-Destino

Dado un conjunto de orígenes y destinos, construiremos la matriz de tiempos de viaje usando OSRM. Luego, integraremos este script con nuestro GeoDataFrame de celdas para construir la Matriz Origen-Destino.

<br>

In [ ]:
import requests
import numpy as np

# ---- funciones auxiliares ----
def obtener_matriz_osrm(origenes, destinos, modo):
    """
    realiza una solicitud a OSRM para obtener la matriz de tiempos de viaje 
    entre 'origenes' y 'destinos'.
    """
    # combinar orígenes y destinos en una sola lista
    coords = origenes + destinos
    coords_str = ';'.join([f"{lon},{lat}" for lon, lat in coords])
    
    # crear índices para OSRM (origen y destino)
    indices_origenes = ';'.join([str(i) for i in range(len(origenes))])
    indices_destinos = ';'.join([str(i) for i in range(len(origenes), len(coords))])

    # definir la url según el modo
    if modo == 'drive':
        url = f"http://localhost:5001/table/v1/driving/{coords_str}"
    else:  # modo == 'walk'
        url = f"http://localhost:5002/table/v1/walking/{coords_str}"
    
    # parámetros para la request
    params = {
        'sources': indices_origenes,
        'destinations': indices_destinos,
        'annotations': 'duration'
    }
    
    # hacer la solicitud a OSRM
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        # 'durations' está en segundos, convertimos a minutos
        durations_seconds = np.array(data['durations'])
        durations_minutes = durations_seconds / 60
        return durations_minutes
    else:
        print(f"Error: {response.status_code}")
        return np.full((len(origenes), len(destinos)), np.inf)
    
def dividir_lista(coordenadas, max_tam):
    """
    generador que divide la lista 'coordenadas' en sub-listas
    de longitud <= max_tam.
    """
    for i in range(0, len(coordenadas), max_tam):
        yield coordenadas[i:i + max_tam] 

In [ ]:
# copiar celdas para orígenes y destinos
gdf_origenes = celdas.copy()
gdf_destinos = celdas.copy()

# convertir a EPSG:32719 y extraer centroides para cada uno, luego pasar a EPSG:4326
gdf_origenes = gdf_origenes.to_crs(32719)
gdf_origenes["geometry"] = gdf_origenes.geometry.centroid
gdf_origenes = gdf_origenes.to_crs(4326)

gdf_destinos = gdf_destinos.to_crs(32719)
gdf_destinos["geometry"] = gdf_destinos.geometry.centroid
gdf_destinos = gdf_destinos.to_crs(4326)

# extraer coordenadas de orígenes y destinos
coords_origenes = [(geom.x, geom.y) for geom in gdf_origenes.geometry]
coords_destinos = [(geom.x, geom.y) for geom in gdf_destinos.geometry]

# definir el tamaño máximo de 'chunk'
max_tam = 1000

# crear una matriz completa llena de infinitos, de tamaño (num_origenes x num_destinos)
matriz_resultado = np.full((len(coords_origenes), len(coords_destinos)), np.inf)

# dividir orígenes y destinos en sub-listas
origen_chunks = list(dividir_lista(coords_origenes, max_tam))
destino_chunks = list(dividir_lista(coords_destinos, max_tam))

# recorrer cada sub-lista de orígenes y destinos para pedir la submatriz a OSRM
for i, sub_origenes in enumerate(origen_chunks):
    for j, sub_destinos in enumerate(destino_chunks):
        
        # llamamos a OSRM para obtener la submatriz
        sub_matriz = obtener_matriz_osrm(sub_origenes, sub_destinos, modo='drive')
        
        # calculamos en qué posiciones de la matriz final se deben ubicar estos valores
        ini_i = i * max_tam
        fin_i = ini_i + len(sub_origenes)
        ini_j = j * max_tam
        fin_j = ini_j + len(sub_destinos)
        
        # insertamos la submatriz en la matriz resultado
        matriz_resultado[ini_i:fin_i, ini_j:fin_j] = sub_matriz

# convertir la matriz (numpy array) en un DataFrame de pandas, con index de orígenes y columns de destinos
drive_matrix = pd.DataFrame(matriz_resultado, index=gdf_origenes.index, columns=gdf_destinos.index)

En este caso, el DataFrame dist_drive es de tamaño $NxN$, donde $N$ es el número de celdas, con valores en minutos. La fila $i$ y columna $j$ representan el tiempo de viaje desde la celda $i$ hasta la celda $j$ en minutos.

In [ ]:
# visualizar las primeras filas y columnas de la matriz
drive_matrix.iloc[:5, :5]

In [ ]:
# copiar celdas para orígenes y destinos
gdf_origenes = celdas.copy()
gdf_destinos = celdas.copy()

# convertir a EPSG:32719 y extraer centroides para cada uno, luego pasar a EPSG:4326
gdf_origenes = gdf_origenes.to_crs(32719)
gdf_origenes["geometry"] = gdf_origenes.geometry.centroid
gdf_origenes = gdf_origenes.to_crs(4326)

gdf_destinos = gdf_destinos.to_crs(32719)
gdf_destinos["geometry"] = gdf_destinos.geometry.centroid
gdf_destinos = gdf_destinos.to_crs(4326)

# extraer coordenadas de orígenes y destinos
coords_origenes = [(geom.x, geom.y) for geom in gdf_origenes.geometry]
coords_destinos = [(geom.x, geom.y) for geom in gdf_destinos.geometry]

# definir el tamaño máximo de 'chunk'
max_tam = 1000

# crear una matriz completa llena de infinitos, de tamaño (num_origenes x num_destinos)
matriz_resultado = np.full((len(coords_origenes), len(coords_destinos)), np.inf)

# dividir orígenes y destinos en sub-listas
origen_chunks = list(dividir_lista(coords_origenes, max_tam))
destino_chunks = list(dividir_lista(coords_destinos, max_tam))

# recorrer cada sub-lista de orígenes y destinos para pedir la submatriz a OSRM
for i, sub_origenes in enumerate(origen_chunks):
    for j, sub_destinos in enumerate(destino_chunks):
        
        # llamamos a OSRM para obtener la submatriz
        sub_matriz = obtener_matriz_osrm(sub_origenes, sub_destinos, modo='walk')
        
        # calculamos en qué posiciones de la matriz final se deben ubicar estos valores
        ini_i = i * max_tam
        fin_i = ini_i + len(sub_origenes)
        ini_j = j * max_tam
        fin_j = ini_j + len(sub_destinos)
        
        # insertamos la submatriz en la matriz resultado
        matriz_resultado[ini_i:fin_i, ini_j:fin_j] = sub_matriz

# convertir la matriz (numpy array) en un DataFrame de pandas, con index de orígenes y columns de destinos
walk_matrix = pd.DataFrame(matriz_resultado, index=gdf_origenes.index, columns=gdf_destinos.index)

---

## 6. Cálculo de la Función de Impedancia

Como se explicó, la accesibilidad gravitacional requiere una función de impedancia que penalice el tiempo o la distancia. A continuación, se presenta un ejemplo bajo una **Función de Impedancia Exponencial**.

In [ ]:
import matplotlib.pyplot as plt

def calcular_impedancia(costo_ij, phi):
    """
    crea una función de impedancia que decae exponencialmente con el costo (distancia/tiempo).
    """
    return np.exp(costo_ij*phi)

# graficar la función de impedancia
tiempos = np.linspace(0, 60, 200)  
phi = -0.1
impedancia_vals = calcular_impedancia(tiempos, phi)

plt.figure(figsize=(8,5))
plt.plot(tiempos, impedancia_vals)
plt.title("Función de Impedancia Exponencial")
plt.xlabel("Tiempo de Viaje (min)")
plt.ylabel("Impedancia")
plt.show()

In [ ]:
# calcular la impedancia para cada matriz
impedance_auto = drive_matrix.map(calcular_impedancia, phi=-0.5)
impedance_caminata = walk_matrix.map(calcular_impedancia, phi=-0.5)

# ver la matriz de impedancias para viajes en auto
impedance_auto.head(5)

In [ ]:
# exportar impedancias (anexo)
# impedance_auto.to_json("anexos/impedance_auto.json")
# impedance_caminata.to_json("anexos/impedance_caminata.json")

---

## 7. Cálculo de Accesibilidades Gravitacionales

Usando la matriz de impedancias calculada previamente $F_{i,j}^{m}$ y las variables de interés $N_{j}^{p}$ (p. ej., los usos de suelo) calculamos las accesibilidades para cada celda y cada variable de interés. Nuestro DataFrame resultante tendrá columnas nuevas, como acc_drive_m2_comercio, acc_drive_m2_industria, etc., representando la accesibilidad desde cada celda a la suma de esos usos de suelo, penalizados por la distancia/tiempo.

<br>

In [ ]:
# filtrar por variables y multiplicación matricial con .dot()
variables = ['m2_comercio', 'm2_departamento', 'n_deporte_y_recreacion', 'n_educacion_y_cultura', 'm2_habitacional', 'm2_industria', 'm2_oficina', 'n_salud', 'n_paraderos']

accesibilidades_auto = impedance_auto.dot(celdas[variables])
accesibilidades_caminata = impedance_caminata.dot(celdas[variables])

# renombrar columnas
accesibilidades_auto.columns = [f"acc_auto_{col}" for col in accesibilidades_auto.columns]
accesibilidades_caminata.columns = [f"acc_caminata_{col}" for col in accesibilidades_caminata.columns]

# agregar geometría
accesibilidades_auto = celdas[['h3', 'geometry']].merge(accesibilidades_auto, left_index=True, right_index=True, how='left')
accesibilidades_caminata = celdas[['h3', 'geometry']].merge(accesibilidades_caminata, left_index=True, right_index=True, how='left')

In [ ]:
# visualizar accesibilidades en automóvil a m2 construidos de comercio
accesibilidades_auto.explore(column='acc_auto_m2_comercio', scheme='NaturalBreaks', k=10)

In [ ]:
# visualizar accesibilidades en caminata a m2 construidos de comercio
accesibilidades_caminata.explore(column='acc_caminata_m2_comercio', scheme='NaturalBreaks', k=10)

---

## 8. Guardar los Datos Procesados

Finalmente, podemos guardar un archivo con las columnas de accesibilidad calculadas. También podríamos combinar ambos resultados en un mismo GeoDataFrame, manteniendo columnas como acc_drive_m2_comercio, acc_walk_m2_comercio, etc., y exportarlo unificado.

<br>

In [ ]:
# exportar los GeoDataFrame de accesibilidades a un archivo GeoJSON
accesibilidades_auto.to_file("celdas_accesibilidades_auto.geojson")
accesibilidades_caminata.to_file("celdas_accesibilidades_caminata.geojson")

**Fin del Laboratorio 3**

---

En el próximo laboratorio, exploraremos cómo estas métricas de accesibilidad pueden integrarse en modelos de localización basados en un enfoque *bid* y *choice*.

### 9. Anexo – Función para Recalcular Accesibilidades

En este laboratorio calculamos matrices de tiempos de viaje (y luego de impedancia) entre todas las celdas, asumiendo una red vial/peatonal fija y centroides de celdas que no cambian. Bajo esta suposición, las matrices de impedancia (`impedance_auto` e `impedance_caminata`) son **siempre las mismas**, incluso si actualizamos los atributos de las celdas (por ejemplo, m2 de comercio, m2 habitacional, etc.), o si trabajamos con distintos cortes temporales (por ejemplo, año 2017 v/s año 2025),

En otras palabras, si la red y los centroides no cambian, solo cambian los metros cuadrados y unidades de los diferentes tipos de uso de suelo ($N_j^p$), no los costos ($C_{i,j}^m$). Por lo tanto, podemos reutilizar las mismas matrices de impedancia una y otra vez para calcular accesibilidades en distintos escenarios, lo que puede ser muy útil para el Laboratorio 5 al momento de generar el motor de simulación.

In [ ]:
# función para crear celdas con información agregada por año
def crear_celdas_con_informacion_por_año(agentes, celdas_area, año):
    agentes = agentes[agentes['año'] <= año]
    celdas_h3_pivot = agentes.pivot_table(
            index='h3',
            columns='destino',
            aggfunc={'superficie_construida': 'sum', 'destino': 'count'},
            fill_value=0).reset_index()

    celdas_h3_pivot.columns = [
        (col[0].replace('superficie_construida', 'm2_') + col[1] if 'superficie_construida' in col[0] else 'n_' + col[1] if col[0] == 'destino' else col[0])
        for col in celdas_h3_pivot.columns
    ]
    celdas_h3_pivot = celdas_area.merge(celdas_h3_pivot, on='h3', how='left')
    celdas_h3_pivot = celdas_h3_pivot.fillna(0)
    celdas_h3_pivot = gpd.GeoDataFrame(celdas_h3_pivot, geometry='geometry')
    return celdas_h3_pivot

# función para calcular accesibilidades
def calcular_accesibilidades(celdas, variables, impedance_matrix, prefix):
    accesibilidades = impedance_matrix.dot(celdas[variables])
    accesibilidades.columns = [f"{prefix}_{col}" for col in accesibilidades.columns]
    accesibilidades = celdas[['h3', 'geometry']].merge(accesibilidades, left_index=True, right_index=True, how='left')
    return accesibilidades

In [ ]:
# ejemplo de uso de la función en dos cortes temporales

# leer impedancias
impedance_auto = pd.read_json("anexos/impedance_auto.json")
impedance_caminata = pd.read_json("anexos/impedance_caminata.json")

# leer celdas_area del Laboratorio 1
celdas_area = gpd.read_file("../lab_01/anexos/celdas_area.geojson")

# leer agentes del Laboratorio 1
agentes = gpd.read_file("../lab_01/anexos/agentes.geojson")

# crear celdas con información agregada por año
celdas_2017 = crear_celdas_con_informacion_por_año(agentes, celdas_area, 2017)
celdas_2023 = crear_celdas_con_informacion_por_año(agentes, celdas_area, 2023)

# definir variables
variables = ['m2_comercio', 'm2_departamento', 'n_deporte_y_recreacion', 'n_educacion_y_cultura', 
             'm2_habitacional_cat1', 'm2_habitacional_cat2', 'm2_habitacional_cat3',
             'm2_industria', 'm2_oficina', 'n_salud']

# calcular accesibilidades por año
accesibilidades_auto_2017 = calcular_accesibilidades(celdas_2017, variables, impedance_auto, prefix='acc_auto')
accesibilidades_caminata_2023 = calcular_accesibilidades(celdas_2023, variables, impedance_caminata, prefix='acc_caminata')